In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, Polygon
import rasterio
from sklearn.model_selection import train_test_split
import os


In [ ]:
def split_train_test(crowns, contour, outfolder, patch_size_m = 25, n_trees_minimum = 10, test_size=0.15, random_state=42):
    """
    Function `split_train_test` takes the following inputs:
    - `crowns` (GeoDataFrame): Polygons representing tree crowns.
    - `contour` (GeoDataFrame): Contours of the area of interest (plantation).
    - `outfolder` (str): Path to the folder where output shapefiles will be saved.
    - `patch_size_m` (float, optional): Size of the square patches in meters. Default is 25.
    - `n_trees_minimum` (int, optional): Minimum number of crowns required to create a square. Default is 15.
    - `test_size` (float, optional): Proportion of the dataset to include in the test split. Default is 0.2.
    - `random_state` (int, optional): Random seed for reproducibility. Default is 42.

    Outputs:
    - `train_squares` (GeoDataFrame): GeoDataFrame containing the training set polygons.
    - `test_squares` (GeoDataFrame): GeoDataFrame containing the test set polygons.

    Steps:
    1. Ensures the output folder exists.
    2. Computes the bounding box of `crowns` and generates a grid of square polygons of fixed size (`patch_size_m`).
    3. Filters squares to retain only those intersecting at least 10 polygons from `crowns`.
    4. Clips the squares with `contour` to ensure they remain within the area of interest.
    5. Splits the resulting squares into training and test sets based on `test_size`.
    6. Saves the training and test sets as shapefiles in the specified `outfolder`.
    7. Returns the training and test GeoDataFrames.
    """
 
    if not os.path.exists(outfolder):
        os.makedirs(outfolder)

    # Créer un nouveau GeoDataFrame pour stocker les polygones carrés qui vont être crées
    squares_gdf = gpd.GeoDataFrame(columns=['geometry'], crs=crowns.crs)

    # Trouver les limites du shapefile crowns
    minx, miny, maxx, maxy = crowns.total_bounds
    # print(f"Bounds: {minx}, {miny}, {maxx}, {maxy}")

    # Calculer le nombre de carrés nécessaires dans les directions x et y    
    num_squares_x = int((maxx - minx) / patch_size_m) + 1
    num_squares_y = int((maxy - miny) / patch_size_m) + 1

    # Créer ces polygones, mais seulement ceux qui contiennent au moins un polygone original
    for i in range(num_squares_x):
        for j in range(num_squares_y):
            square = box(minx + i * patch_size_m, miny + j * patch_size_m, minx + (i + 1) * patch_size_m, miny + (j + 1) * patch_size_m)
        
            # Vérifier si le carré intersecte au moins n_trees_minimum polygones de crowns
            if crowns.intersects(square).sum() >= n_trees_minimum:
                squares_gdf = pd.concat([squares_gdf, gpd.GeoDataFrame({'geometry': [square]}, crs=crowns.crs)], ignore_index=True)
    
    #s'assurer que le contour et les carrés ont la même projection
    contour = contour.to_crs(squares_gdf.crs)

    # Découper les carrés avec le contour
    squares_gdf = gpd.overlay(squares_gdf, contour, how='intersection')

    # split traintest
    test_percent = test_size
    train_percent = 1 - test_percent
    
    # Split the squares into train and test sets
    train_squares, test_squares = train_test_split(squares_gdf, test_size=test_percent, random_state=random_state)

    #Calculer le nombre de carrés et l'aire totale dans chaque ensemble
    num_train_squares = len(train_squares)
    num_test_squares = len(test_squares)
    train_area = train_squares.geometry.area.sum()
    test_area = test_squares.geometry.area.sum()
    print(f"Nombre total de carrés: {len(squares_gdf)}")
    print(f"Nombre de carrés d'entraînement: {num_train_squares}")
    print(f"Nombre de carrés de test: {num_test_squares}")
    print(f"Aire totale des carrés d'entraînement: {train_area:.2f} m²")
    print(f"Aire totale des carrés en test: {test_area:.2f}")
    print(f"Proportion de la surface en test: {test_area/(train_area+test_area):.2f}")


    # Enregistrer les GeoDataFrames dans trois fichiers shapefile distincts
    train_squares.to_file(os.path.join(outfolder, 'train_areas.shp'), driver='ESRI Shapefile')
    test_squares.to_file(os.path.join(outfolder, 'test_areas.shp'), driver='ESRI Shapefile')

    print("Nouveau shapefiles 'train_areas.shp', 'test_areas.shp' créé avec succès.")
    print(f"outfiles: {os.path.join(outfolder, 'train_areas.shp')}, {os.path.join(outfolder, 'test_areas.shp')}")

    return train_squares, test_squares

## Cw_JR_East

In [17]:
crowns = gpd.read_file(r'C:\Git\data_glo7030-projet\Cw_JR_East\Crowns_clean\Cw_JR_East_crowns_clean_v2.shp')
contour = gpd.read_file(r'C:\Git\data_glo7030-projet\Cw_JR_East\Contour\Cw_JR_East_Contour.shp')
outfolder = r'C:\Git\data_glo7030-projet\Cw_JR_East\train_test_split'
train, test = split_train_test(crowns, contour, outfolder, patch_size_m = 30, test_size=0.10, random_state=42)

Nombre de carrés créés: 33
Nombre total de carrés: 35
Nombre de carrés d'entraînement: 31
Nombre de carrés de test: 4
Aire totale des carrés d'entraînement: 16412.09 m²
Aire totale des carrés en test: 1570.30
Proportion de la surface en test: 0.09 m²
Nouveau shapefiles 'train_areas.shp', 'test_areas.shp' créé avec succès.
outfiles: C:\Git\data_glo7030-projet\Cw_JR_East\train_test_split\train_areas.shp, C:\Git\data_glo7030-projet\Cw_JR_East\train_test_split\test_areas.shp


## Fdc_JR_W45

In [19]:
crowns = gpd.read_file(r'C:\Git\data_glo7030-projet\Fdc_JR_W45\Crowns_clean\Fdc_JR_W45_crowns_clean_v2.shp')
contour = gpd.read_file(r'C:\Git\data_glo7030-projet\Fdc_JR_W45\Contour\Fdc_JR_W45_2023_contour_precis.shp')
outfolder = r'C:\Git\data_glo7030-projet\Fdc_JR_W45\train_test_split'
train, test = split_train_test(crowns, contour, outfolder, patch_size_m = 30, test_size=0.10, random_state=42)

Nombre de carrés créés: 18
Nombre total de carrés: 18
Nombre de carrés d'entraînement: 16
Nombre de carrés de test: 2
Aire totale des carrés d'entraînement: 8819.01 m²
Aire totale des carrés en test: 889.31
Proportion de la surface en test: 0.09 m²
Nouveau shapefiles 'train_areas.shp', 'test_areas.shp' créé avec succès.
outfiles: C:\Git\data_glo7030-projet\Fdc_JR_W45\train_test_split\train_areas.shp, C:\Git\data_glo7030-projet\Fdc_JR_W45\train_test_split\test_areas.shp


## Sortir une image de test à envoyer en inférence

In [ ]:
from rasterio.mask import mask

# Ouvrir image de Ground Truth
with rasterio.open(r"C:\Git\data_glo7030-projet\Fdc_JR_W45\MS_2023_03_21\Fdc_JR_W45_2023_03_21_panIMG_0037_6_MS_alP1_cor2-001.tif") as src:
    print(f"Dimensions: {src.shape}")  
    print(f"Nombre de bandes : {src.count}")
    print(f"Projection : {src.crs}")
    img = src.read(1)

    # Mask the image using the test GeoDataFrame
    
    # Simplify the test geometries to reduce memory usage
    #simplified_test = test.geometry.simplify(tolerance=1.0, preserve_topology=True)
    
    print("Creating mask for test polygons...")
    #test_masked, transform = mask(src, simplified_test, crop=False, nodata=0) #for the simplified test polygons
    test_masked, transform = mask(src, test.geometry, crop=False, nodata=0)

    # Replace values outside the polygons with 0 for all bands
    print("Replacing values outside polygons with 0...")
    masked_image = np.where(test_masked == 0, 0, test_masked)

    # Save the masked image for further use
    print("Masked image created with regions outside polygons set to 0.")
    print("Saving masked image...")
    # Save the masked image as a new GeoTIFF file
    output_path = r"C:\Git\data_glo7030-projet\Fdc_JR_W45\MS_2023_03_21\masked_image.tif"
    with rasterio.open(output_path, 'w', driver='GTiff', height=masked_image.shape[1], width=masked_image.shape[2],
                       count=masked_image.shape[0], dtype=masked_image.dtype, crs=src.crs, transform=transform) as dst:
        for i in range(masked_image.shape[0]):
            dst.write(masked_image[i], i + 1)
    print(f"Masked image saved to {output_path}")

    
    

    # Close the source image
    #src.close()

    






Dimensions: (11339, 11060)
Nombre de bandes : 10
Projection : EPSG:26910
Creating mask for test polygons...
Replacing values outside polygons with 0...
Masked image created with regions outside polygons set to 0.
Saving masked image...


RasterioIOError: masked_image.tif: Free disk space available is 120868864 bytes, whereas 5016373600 are at least necessary. You can disable this check by defining the CHECK_DISK_FREE_SPACE configuration option to FALSE.

In [21]:
## Version dans laquelle je crée un fichier GeoTIFF pour chaque polygone

# Ouvrir image de Ground Truth
with rasterio.open(r"C:\Git\data_glo7030-projet\Fdc_JR_W45\MS_2023_03_21\Fdc_JR_W45_2023_03_21_panIMG_0037_6_MS_alP1_cor2-001.tif") as src:
    print(f"Dimensions: {src.shape}")  
    print(f"Nombre de bandes : {src.count}")
    print(f"Projection : {src.crs}")
    img = src.read(1)

# Loop through each polygon in the test GeoDataFrame
for idx, polygon in enumerate(test.geometry):
    print(f"Processing polygon {idx + 1}/{len(test)}...")
        
    # Mask the image for the current polygon
    single_polygon_masked, single_transform = mask(src, [polygon], crop=True, nodata=0)
        
    # Replace values outside the polygon with 0
    single_masked_image = np.where(single_polygon_masked == 0, 0, single_polygon_masked)
        
    # Save the masked image for the current polygon
    single_output_path = os.path.join(outfolder, f"masked_image_polygon_{idx + 1}.tif")
    with rasterio.open(single_output_path, 'w', driver='GTiff', height=single_masked_image.shape[1], width=single_masked_image.shape[2],
                        count=single_masked_image.shape[0], dtype=single_masked_image.dtype, crs=src.crs, transform=single_transform) as dst:
        for i in range(single_masked_image.shape[0]):
            dst.write(single_masked_image[i], i + 1)
    print(f"Masked image for polygon {idx + 1} saved to {single_output_path}")

Dimensions: (11339, 11060)
Nombre de bandes : 10
Projection : EPSG:26910
Processing polygon 1/2...


RasterioIOError: Dataset is closed: C:\Git\data_glo7030-projet\Fdc_JR_W45\MS_2023_03_21\Fdc_JR_W45_2023_03_21_panIMG_0037_6_MS_alP1_cor2-001.tif